# 1b - CIDER VDJ Output by Read-Confidence Filter

**Purpose**: Build two alternate `cider_base` variants that restrict CDR3
calls to a single CIDER `filter` QC category, for a response-comparison
robustness check (`1_figures/17_response_by_read_confidence.ipynb`):

- **PASS only** — high-confidence CDR3 calls (`filter == "PASS"`)
- **Low-confidence only** — `filter == "NO_HIGH_QUAL_SUPPORT"`

The primary pipeline (`1_cider.ipynb` → `cider_base.csv`) pools both
categories together; that variant is *not* regenerated here.

Both variants reuse `build_cider_base()` (`helpers/shortcuts.r`) — the same
function `1_cider.ipynb` uses — so the grouping, summarising, and zero-fill
logic (samples where CIDER ran but nothing passed the filter get `0`,
rather than being silently dropped) is identical across all three variants.

| | |
|---|---|
| **Inputs** | `CIDER_DIR/cider.vdj.csv`, `READY_DIR/clinical_update.txt`, `PURPLE_DIR/purity.csv` |
| **Outputs** | `CIDER_PROJ_DIR/cider_base_pass_only.csv`, `cider_base_low_confidence_only.csv` |
| **Run before** | `1_figures/17_response_by_read_confidence.ipynb` |


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))

assert_files_exist(
  "CIDER VDJ file" = paste0(CIDER_DIR, "cider.vdj.csv")
)


# 0 - Raw CIDER output and valid sample set

In [ ]:
# Raw CIDER VDJ output, unfiltered — build_cider_base() applies the `filter`
# selection itself, once per variant below.
cider_vdj_raw <- fread(paste0(CIDER_DIR, "cider.vdj.csv"))


In [ ]:
# all_samples: samples that satisfy all three conditions (identical derivation
# to 1_cider.ipynb — see there for full rationale):
#   1. Present in clinical_update.txt   — have clinical data
#   2. Present in PURPLE_DIR/purity.csv — WGS was run
#   3. Present in raw CIDER output      — CIDER was run (at least one call,
#                                         even if all failed QC filters)
# Independent of which `filter` values a variant keeps, since it's about
# whether CIDER ran at all.
clinical_ids <- fread(paste0(READY_DIR, "clinical_update.txt")) |>
  select(sampleId) |>
  distinct()

wgs_ids <- fread(paste0(PURPLE_DIR, "purity.csv")) |>
  select(sampleId) |>
  distinct()

cider_ids <- cider_vdj_raw |>
  select(sampleId) |>
  distinct()

all_samples <- clinical_ids |>
  inner_join(wgs_ids,   by = "sampleId") |>
  inner_join(cider_ids, by = "sampleId")


# 1 - Build and save each variant

In [ ]:
cider_base_pass_only <- build_cider_base(
  cider_vdj_raw,
  filter_values = c("PASS"),
  all_samples   = all_samples
)
fwrite(cider_base_pass_only, paste0(CIDER_PROJ_DIR, "cider_base_pass_only.csv"))


In [ ]:
cider_base_low_confidence_only <- build_cider_base(
  cider_vdj_raw,
  filter_values = c("NO_HIGH_QUAL_SUPPORT"),
  all_samples   = all_samples
)
fwrite(cider_base_low_confidence_only, paste0(CIDER_PROJ_DIR, "cider_base_low_confidence_only.csv"))


In [ ]:
message(
  "Saved: cider_base_pass_only.csv, cider_base_low_confidence_only.csv\n",
  "Note: the combined PASS + NO_HIGH_QUAL_SUPPORT variant is the primary ",
  "pipeline output (cider_base.csv, from 1_cider.ipynb) and is not regenerated here."
)
